> **Public portfolio copy:** Saved execution outputs have been removed. The notebook source code and explanations are retained. Running the notebook requires a separately authorised DICOM input file in the local `Data/` folder. No Stanford COCA image or dataset file is included in this repository.

# Medical Image Viewer Prototype

**Project 2: DICOM Extension – Medical Image Viewer and Preprocessing**

## Overview

This notebook develops an interactive CT image viewer prototype using Python, Matplotlib and ipywidgets.

The prototype integrates DICOM image loading, Hounsfield Unit (HU) conversion, CT windowing, image filtering, contrast enhancement and interactive image selection.

## Objectives

- Develop an interactive interface for viewing CT images.
- Implement standard CT window presets and custom window controls.
- Explore image filtering and contrast enhancement techniques.
- Integrate image-processing operations into an interactive viewer.

## Dataset

The prototype uses a single CT DICOM image from the project's `Data` directory.

## Technologies

Python, pydicom, NumPy, Matplotlib, SciPy, scikit-image and ipywidgets.

## 1. Environment Setup and DICOM Image Loading

This section imports the required libraries, establishes the project data paths, loads the DICOM image and converts its pixel values into Hounsfield Units (HU).

In [ ]:
# IMPORT LIBRARIES FOR THE MEDICAL IMAGE VIEWER PROTOTYPE

# File and path management
from pathlib import Path

# Numerical processing
import numpy as np

# DICOM handling
import pydicom

# Image visualization
import matplotlib.pyplot as plt

# Interactive widgets
import ipywidgets as widgets
from IPython.display import display, clear_output

# Image processing
from scipy.ndimage import gaussian_filter
from skimage.transform import resize
from skimage.exposure import equalize_adapthist

# Define project and data paths
project_path = Path.cwd().parent
data_path = project_path / "Data"

# Load DICOM image
dicom_file = data_path / "IM-0001-0001.dcm"
dcm = pydicom.dcmread(dicom_file)

# Extract pixel array
pixel_array = dcm.pixel_array

# Convert pixel values to Hounsfield Units
ct_image = (
    pixel_array * float(dcm.RescaleSlope)
    + float(dcm.RescaleIntercept)
)

# Display image information
print("DICOM loaded successfully.")
print("Image dimensions:", ct_image.shape)
print("Minimum HU:", ct_image.min())
print("Maximum HU:", ct_image.max())

## 2. CT Windowing and Interactive Window Controls

This section implements a CT windowing function, demonstrates standard window presets and introduces interactive Window Centre (WC) and Window Width (WW) controls.

In [ ]:
# DEFINE THE CT WINDOWING FUNCTION

def apply_window(image, window_center, window_width):

    # Calculate lower and upper window boundaries
    lower = window_center - (window_width / 2)
    upper = window_center + (window_width / 2)

    # Clip image values to the selected window
    windowed_image = np.clip(image, lower, upper)

    # Normalize pixel values to the range 0–1
    windowed_image = (
        (windowed_image - lower) / (upper - lower)
    )

    return windowed_image

In [ ]:
# APPLY STANDARD CT WINDOW PRESETS

# Define commonly used CT window settings
window_presets = {
    "Soft Tissue": (40, 400),
    "Lung": (-600, 1500),
    "Bone": (500, 2000)
}

# Create a figure for the three window displays
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Apply and display each window preset
for ax, (name, (wc, ww)) in zip(axes, window_presets.items()):

    # Apply the selected window
    windowed_image = apply_window(ct_image, wc, ww)

    # Display the windowed image
    ax.imshow(windowed_image, cmap="gray")
    ax.set_title(f"{name}\nWC={wc}, WW={ww}")
    ax.axis("off")

# Adjust layout and display
plt.tight_layout()
plt.show()

In [ ]:
# CREATE INTERACTIVE WINDOW LEVEL CONTROLS

import ipywidgets as widgets
from IPython.display import display

# Create sliders for Window Centre and Window Width
wc_slider = widgets.IntSlider(
    value=40,
    min=-1000,
    max=1000,
    step=10,
    description="Window Centre:"
)

ww_slider = widgets.IntSlider(
    value=400,
    min=1,
    max=3000,
    step=10,
    description="Window Width:"
)

# Create a button to apply the selected settings
apply_button = widgets.Button(
    description="Apply Window",
    button_style="primary"
)

# Create an output area for the image
output = widgets.Output()

# Define the button action
def update_window(button):

    # Clear the previous image
    output.clear_output(wait=True)

    # Apply the selected window settings
    windowed_image = apply_window(
        ct_image,
        wc_slider.value,
        ww_slider.value
    )

    # Display the updated image
    with output:
        plt.figure(figsize=(7, 7))
        plt.imshow(windowed_image, cmap="gray")
        plt.title(
            f"Interactive CT Viewer | WC={wc_slider.value}, "
            f"WW={ww_slider.value}"
        )
        plt.axis("off")
        plt.show()

# Connect the button to the update function
apply_button.on_click(update_window)

# Display the controls
display(wc_slider, ww_slider, apply_button, output)

## 3. Image Filtering and Noise Reduction

This section applies median and Gaussian filtering to the CT image and compares their effects on image smoothing and noise reduction.

In [ ]:
# IMPORT THE MEDIAN FILTER FUNCTION

from scipy.ndimage import median_filter

# Apply a 3 × 3 median filter to the CT image
filtered_image = median_filter(ct_image, size=3)

# Display original and filtered images side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Display original CT image
axes[0].imshow(ct_image, cmap="gray", vmin=-200, vmax=300)
axes[0].set_title("Original CT Image")
axes[0].axis("off")

# Display median-filtered CT image
axes[1].imshow(filtered_image, cmap="gray", vmin=-200, vmax=300)
axes[1].set_title("Median Filtered CT Image")
axes[1].axis("off")

# Display comparison
plt.tight_layout()
plt.show()

In [ ]:
# IMPORT GAUSSIAN FILTER

from scipy.ndimage import gaussian_filter

# Apply Gaussian smoothing to the original CT image
gaussian_image = gaussian_filter(ct_image, sigma=1)

# Compare original, median-filtered and Gaussian-filtered images
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Display original image
axes[0].imshow(ct_image, cmap="gray", vmin=-200, vmax=300)
axes[0].set_title("Original CT Image")
axes[0].axis("off")

# Display median-filtered image
axes[1].imshow(filtered_image, cmap="gray", vmin=-200, vmax=300)
axes[1].set_title("Median Filter")
axes[1].axis("off")

# Display Gaussian-filtered image
axes[2].imshow(gaussian_image, cmap="gray", vmin=-200, vmax=300)
axes[2].set_title("Gaussian Filter")
axes[2].axis("off")

# Display comparison
plt.tight_layout()
plt.show()

## 4. Contrast Enhancement and Histogram Equalization

This section explores global histogram equalization and Contrast-Limited Adaptive Histogram Equalization (CLAHE) to demonstrate different approaches to CT image contrast enhancement.

In [ ]:
# IMPORT HISTOGRAM EQUALIZATION FUNCTION

from skimage import exposure

# Normalize the CT image to a 0–1 range for enhancement
normalized_image = exposure.rescale_intensity(
    ct_image,
    in_range=(-200, 300),
    out_range=(0, 1)
)

# Apply histogram equalization
equalized_image = exposure.equalize_hist(normalized_image)

# Compare original and enhanced images
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Display original CT image
axes[0].imshow(normalized_image, cmap="gray")
axes[0].set_title("Original CT Image")
axes[0].axis("off")

# Display histogram-equalized image
axes[1].imshow(equalized_image, cmap="gray")
axes[1].set_title("Histogram Equalization")
axes[1].axis("off")

# Display comparison
plt.tight_layout()
plt.show()

In [ ]:
# APPLY CONTRAST-LIMITED ADAPTIVE HISTOGRAM EQUALIZATION (CLAHE)

# Apply CLAHE to the normalized CT image
clahe_image = exposure.equalize_adapthist(
    normalized_image,
    clip_limit=0.03
)

# Compare global and adaptive histogram equalization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Display original image
axes[0].imshow(normalized_image, cmap="gray")
axes[0].set_title("Original CT Image")
axes[0].axis("off")

# Display global histogram equalization
axes[1].imshow(equalized_image, cmap="gray")
axes[1].set_title("Global Histogram Equalization")
axes[1].axis("off")

# Display CLAHE result
axes[2].imshow(clahe_image, cmap="gray")
axes[2].set_title("CLAHE Enhancement")
axes[2].axis("off")

# Display comparison
plt.tight_layout()
plt.show()

## 5. Interactive Medical Image Viewer

This section integrates the available image-processing operations into an interactive viewer, allowing image selection and custom CT windowing through graphical controls.

In [ ]:
# CREATE AN INTERACTIVE CT IMAGE PROCESSING SELECTOR

import ipywidgets as widgets
from IPython.display import display, clear_output

# Prepare the available image processing options
processing_options = {
    "Original CT Image": ct_image,
    "Soft Tissue Window": apply_window(ct_image, 40, 400),
    "Gaussian Filter": gaussian_image,
    "Histogram Equalization": equalized_image,
    "CLAHE Enhancement": clahe_image
}

# Create a dropdown menu
image_selector = widgets.Dropdown(
    options=list(processing_options.keys()),
    value="Original CT Image",
    description="Image:"
)

# Create an output area
image_output = widgets.Output()

# Define the display function
def display_selected_image(change):
    with image_output:
        clear_output(wait=True)

        # Retrieve the selected image
        selected_image = processing_options[change["new"]]

        # Display the selected image
        plt.figure(figsize=(7, 7))
        plt.imshow(selected_image, cmap="gray")
        plt.title(change["new"])
        plt.axis("off")
        plt.show()

# Connect dropdown to display function
image_selector.observe(display_selected_image, names="value")

# Display the controls and initial image
display(image_selector, image_output)

# Display the initial selection
with image_output:
    plt.figure(figsize=(7, 7))
    plt.imshow(ct_image, cmap="gray")
    plt.title("Original CT Image")
    plt.axis("off")
    plt.show()

In [ ]:
# CREATE A BUTTON-CONTROLLED INTERACTIVE CT VIEWER

# Create the image selection dropdown
image_selector = widgets.Dropdown(
    options=list(processing_options.keys()) + ["Custom Window"],
    value="Original CT Image",
    description="Image:"
)

# Create the Window Centre slider
wc_slider = widgets.IntSlider(
    value=40,
    min=-1000,
    max=1000,
    step=10,
    description="Window Centre:"
)

# Create the Window Width slider
ww_slider = widgets.IntSlider(
    value=400,
    min=1,
    max=3000,
    step=10,
    description="Window Width:"
)

# Create the Apply button
apply_button = widgets.Button(
    description="Apply",
    button_style="primary"
)

# Create the image output area
image_output = widgets.Output()


# Define the function that displays the selected image
def update_viewer(button=None):

    # Clear the previous image
    with image_output:
        clear_output(wait=True)

        # Check whether custom windowing was selected
        if image_selector.value == "Custom Window":

            # Read the selected window settings
            wc = wc_slider.value
            ww = ww_slider.value

            # Calculate the window display limits
            lower_limit = wc - ww / 2
            upper_limit = wc + ww / 2

            # Display the original HU image using the selected window
            plt.figure(figsize=(7, 7))
            plt.imshow(
                ct_image,
                cmap="gray",
                vmin=lower_limit,
                vmax=upper_limit
            )

            # Display the selected window settings
            plt.title(f"Custom Window | WC={wc}, WW={ww}")

        else:

            # Retrieve the selected processed image
            selected_image = processing_options[image_selector.value]

            # Display the selected image
            plt.figure(figsize=(7, 7))
            plt.imshow(selected_image, cmap="gray")
            plt.title(image_selector.value)

        # Remove axes and display the image
        plt.axis("off")
        plt.show()


# Connect the Apply button to the display function
apply_button.on_click(update_viewer)

# Display the viewer controls
display(
    image_selector,
    wc_slider,
    ww_slider,
    apply_button,
    image_output
)

# Display the initial CT image
update_viewer()

## Prototype Summary

This notebook implements an interactive CT image viewer prototype using Python, Matplotlib and ipywidgets.

### Implemented Features
- DICOM image loading and pixel extraction.
- Conversion of raw pixel values to Hounsfield Units (HU).
- Display of the original CT image.
- Selection of processed images from a dropdown menu.
- Interactive Window Centre (WC) and Window Width (WW) controls.
- Custom CT windowing applied through a button.
- Image processing techniques including median filtering, Gaussian filtering and histogram equalization.

### Current Limitations
- The prototype currently operates on a single CT image rather than a complete DICOM series.
- It is a Jupyter Notebook prototype, not a standalone medical imaging application.
- Additional DICOM viewer features, including slice navigation and multiplanar reconstruction, remain future improvements.